# 04 · Analysis

Summarize the answers saved by `03_rag_queries.ipynb`: which risk factors the model raised, which sources it relied on, and how well the answers cover the factors in the evidence cards.

No API key is needed for this notebook.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from src.config import EVIDENCE_FILE, RESULTS_FILE

assert RESULTS_FILE.exists(), "Run 03_rag_queries.ipynb first."
results = json.loads(RESULTS_FILE.read_text(encoding="utf-8"))
cards = json.loads(EVIDENCE_FILE.read_text(encoding="utf-8"))
print(f"{len(results)} answers loaded; {sum(r['answer'] is not None for r in results)} parsed as JSON")

## Answers at a glance

In [ ]:
def as_text(items):
    """Render list items (strings or dicts) as short strings."""
    return [i if isinstance(i, str) else json.dumps(i, ensure_ascii=False) for i in (items or [])]


summary = pd.DataFrame(
    [
        {
            "question": r["question"][:60] + "...",
            "risk_factors": len((r["answer"] or {}).get("risk_factors", [])),
            "exposures": len((r["answer"] or {}).get("environmental_exposures", [])),
            "relative_risk": (r["answer"] or {}).get("relative_risk", ""),
        }
        for r in results
    ]
)
pd.set_option("display.max_colwidth", 200)
summary

## Sources retrieved

In [ ]:
sources = pd.DataFrame([s for r in results for s in r["sources"]])
counts = sources.groupby(["type", "source"]).size().sort_values(ascending=True)
counts.index = [f"{t}: {s[:50]}" for t, s in counts.index]

counts.plot.barh(figsize=(8, 3), title="Times each document was retrieved")
plt.xlabel("retrievals")
plt.tight_layout()
plt.show()

## Coverage of evidence-card factors

A factor counts as covered when its name (or the first word of a compound name) appears anywhere in an answer.

In [ ]:
answer_text = " ".join(
    item
    for r in results
    for v in (r["answer"] or {}).values()
    for item in (as_text(v) if isinstance(v, list) else [str(v)])
).lower()

rows = []
for card in cards:
    for f in card["factors"]:
        name = f["factor"].lower()
        keys = {name, name.split(" / ")[0], name.split()[0]}
        rows.append(
            {
                "factor": f["factor"],
                "category": f["category"],
                "card_year": card["year"],
                "mentioned": any(k in answer_text for k in keys),
            }
        )

coverage = pd.DataFrame(rows)
print(f"Covered {coverage.mentioned.sum()} of {len(coverage)} evidence factors")
coverage.sort_values(["mentioned", "category"], ascending=[False, True])

## All risk factors raised

In [ ]:
for r in results:
    print(f"\n=== {r['question']}")
    for item in as_text((r["answer"] or {}).get("risk_factors")):
        print("  -", item)